# 🧠 Experimentos Deep Learning - Sinal Raw

## Objetivo
Avaliar modelos de Deep Learning usando o sinal raw (sem pré-processamento wavelet):
- **CNN** (Convolutional Neural Network)
- **LSTM** (Long Short-Term Memory)
- **CNN-LSTM** (Híbrido)
- **Transformer**

## Pipeline
1. Carregar dados
2. Preparar para DL (adicionar dimensão de canal)
3. Treinar cada modelo com early stopping
4. Avaliar no conjunto de teste
5. Comparar resultados

In [1]:
# ── GPU selection (must come BEFORE importing TensorFlow) ──
import os, sys
sys.path.append('.')
from config.experiment_config import (
    DATA_DIR, RESULTS_DIR, MODELS_DIR,
    DL_TRAINING_CONFIG, DL_MODELS_CONFIG,
    generate_dl_grid, SEED, GPU_ID, EPOCHS_OVERRIDE, MAX_GRID_CONFIGS,
)
# (CUDA_VISIBLE_DEVICES já foi configurado pelo experiment_config)

# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import random
import warnings
warnings.filterwarnings('ignore')

# TensorFlow (importado APÓS seleção de GPU)
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU disponível: {tf.config.list_physical_devices('GPU')}")

# Imports locais
from src.models import (
    create_cnn_model, create_lstm_model, 
    create_cnn_lstm_model, create_transformer_model,
    get_callbacks, get_distribute_strategy
)
from src.evaluation import RegressionEvaluator, ResultsManager
from src.visualization import ExperimentVisualizer

# ── Reprodutibilidade: fixar seed global ──
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# Multi-GPU strategy
strategy = get_distribute_strategy()

# Configuração
plt.style.use('seaborn-v0_8-whitegrid')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / "dl_raw_experiments").mkdir(exist_ok=True)

print(f"\n✅ Imports realizados com sucesso!")
print(f"🎲 SEED={SEED} definido para numpy, tf e random")
if GPU_ID:
    print(f"🖥️  GPU selecionada: {GPU_ID}")
if EPOCHS_OVERRIDE:
    print(f"⚡ EPOCHS_OVERRIDE={EPOCHS_OVERRIDE}")
if MAX_GRID_CONFIGS:
    print(f"⚡ MAX_GRID_CONFIGS={MAX_GRID_CONFIGS}")

I0000 00:00:1777262282.338836 1018288 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow version: 2.21.0
GPU disponível: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


⚡ OneDeviceStrategy: GPU:0

✅ Imports realizados com sucesso!
🎲 SEED=42 definido para numpy, tf e random
🖥️  GPU selecionada: 0


I0000 00:00:1777262284.315385 1018288 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22289 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:01:00.0, compute capability: 8.9


## 1. Carregar e Preparar Dados

In [2]:
# Carregar datasets
X_train = np.load(DATA_DIR / "X_train.npy")
y_train = np.load(DATA_DIR / "y_train.npy")
X_val = np.load(DATA_DIR / "X_val.npy")
y_val = np.load(DATA_DIR / "y_val.npy")
X_test = np.load(DATA_DIR / "X_test.npy")
y_test = np.load(DATA_DIR / "y_test.npy")

# Adicionar dimensão de canal para CNN/LSTM
X_train = X_train[..., np.newaxis]  # (N, seq_len, 1)
X_val = X_val[..., np.newaxis]
X_test = X_test[..., np.newaxis]

print(f"📦 Dados Carregados (com canal):")
print(f"  Train: X={X_train.shape}, y={y_train.shape}")
print(f"  Val:   X={X_val.shape}, y={y_val.shape}")
print(f"  Test:  X={X_test.shape}, y={y_test.shape}")

input_shape = X_train.shape[1:]
print(f"\nInput shape para modelos: {input_shape}")

📦 Dados Carregados (com canal):
  Train: X=(34820, 256, 1), y=(34820,)
  Val:   X=(7462, 256, 1), y=(7462,)
  Test:  X=(7462, 256, 1), y=(7462,)

Input shape para modelos: (256, 1)


## 2. Configuração

In [3]:
# Gerenciadores
results_manager = ResultsManager(RESULTS_DIR / "dl_raw_experiments")
evaluator = RegressionEvaluator()
visualizer = ExperimentVisualizer()

# Configuração de treinamento (do config centralizado)
training_config = DL_TRAINING_CONFIG.copy()
print("Configuração de Treinamento:")
for k, v in training_config.items():
    print(f"  {k}: {v}")

# Armazenar resultados
all_results = {}          # melhor de cada arquitetura
all_histories = {}        # histórico do melhor
all_grid_results = []     # TODOS os resultados do grid (todas combinações)

Configuração de Treinamento:
  epochs: 100
  batch_size: 64
  early_stopping_patience: 15
  reduce_lr_patience: 7
  reduce_lr_factor: 0.5
  min_lr: 1e-06
  verbose: 1


## 3. Experimento 1: CNN

In [4]:
print("="*70)
print("🔵 Grid Search: CNN com Sinal Raw")
print("="*70)

grid = generate_dl_grid('CNN')
base_params = DL_MODELS_CONFIG['CNN'].copy()
best_rmse, best_key = float('inf'), None

for gi, variation in enumerate(grid):
    params = {**base_params, **variation}
    run_name = f"Raw_CNN_g{gi}"
    print(f"\n--- [{gi+1}/{len(grid)}] {run_name}: {variation}")

    tf.keras.backend.clear_session()
    with strategy.scope():
        model = create_cnn_model(input_shape, params=params)

    model_path = str(MODELS_DIR / f"raw_cnn_g{gi}.keras")
    callbacks = get_callbacks(
        model_path,
        patience_early=training_config['early_stopping_patience'],
        patience_lr=training_config['reduce_lr_patience'],
        min_lr=training_config['min_lr'],
    )

    t0 = time.time()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=training_config['epochs'],
        batch_size=training_config['batch_size'],
        callbacks=callbacks, verbose=0,
    )
    elapsed = time.time() - t0

    y_pred = model.predict(X_test, verbose=0).flatten()
    metrics = evaluator.evaluate(y_test, y_pred)

    print(f"    RMSE={metrics['rmse']:.6f}  MAE={metrics['mae']:.6f}  "
          f"R²={metrics['r2']:.6f}  Time={elapsed:.1f}s")

    row = {'Model': 'Raw_CNN', 'grid_idx': gi, **variation,
           'RMSE': metrics['rmse'], 'MAE': metrics['mae'],
           'R²': metrics['r2'], 'Params': model.count_params(),
           'Time (s)': elapsed, 'Epochs': len(history.history['loss'])}
    all_grid_results.append(row)

    if metrics['rmse'] < best_rmse:
        best_rmse = metrics['rmse']
        best_key = run_name
        all_results['Raw_CNN'] = {
            'metrics': metrics, 'time': elapsed,
            'epochs': len(history.history['loss']),
            'y_pred': y_pred, 'params': model.count_params(),
            'best_variation': variation,
        }
        all_histories['Raw_CNN'] = history.history
        model.save(str(MODELS_DIR / "raw_cnn_best.keras"))

    results_manager.log_experiment('DL_Raw', run_name, metrics, {'params': params})

print(f"\n🏆 Melhor CNN: {best_key} — RMSE={best_rmse:.6f}")

🔵 Grid Search: CNN com Sinal Raw

--- [1/36] Raw_CNN_g0: {'dropout_rate': 0.2, 'l2_reg': 0.0001, 'filters': [32, 64, 128], 'kernel_sizes': [7, 5, 3]}


I0000 00:00:1777262285.708027 1018288 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.


I0000 00:00:1777262286.777901 1018957 cuda_dnn.cc:461] Loaded cuDNN version 91900



Epoch 26: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.



Epoch 41: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.


...


Epoch 34: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.



Epoch 41: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.


Epoch 42: early stopping


Restoring model weights from the end of the best epoch: 27.


    RMSE=0.252956  MAE=0.197092  R²=0.936683  Time=231.1s

🏆 Melhor CNN: Raw_CNN_g9 — RMSE=0.232513


## 4. Experimento 2: LSTM

In [5]:
print("="*70)
print("🔵 Grid Search: LSTM com Sinal Raw")
print("="*70)

grid = generate_dl_grid('LSTM')
base_params = DL_MODELS_CONFIG['LSTM'].copy()
best_rmse, best_key = float('inf'), None

for gi, variation in enumerate(grid):
    params = {**base_params, **variation}
    run_name = f"Raw_LSTM_g{gi}"
    print(f"\n--- [{gi+1}/{len(grid)}] {run_name}: {variation}")

    tf.keras.backend.clear_session()
    with strategy.scope():
        model = create_lstm_model(input_shape, params=params)

    model_path = str(MODELS_DIR / f"raw_lstm_g{gi}.keras")
    callbacks = get_callbacks(
        model_path,
        patience_early=training_config['early_stopping_patience'],
        patience_lr=training_config['reduce_lr_patience'],
        min_lr=training_config['min_lr'],
    )

    t0 = time.time()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=training_config['epochs'],
        batch_size=training_config['batch_size'],
        callbacks=callbacks, verbose=0,
    )
    elapsed = time.time() - t0

    y_pred = model.predict(X_test, verbose=0).flatten()
    metrics = evaluator.evaluate(y_test, y_pred)

    print(f"    RMSE={metrics['rmse']:.6f}  MAE={metrics['mae']:.6f}  "
          f"R²={metrics['r2']:.6f}  Time={elapsed:.1f}s")

    row = {'Model': 'Raw_LSTM', 'grid_idx': gi, **variation,
           'RMSE': metrics['rmse'], 'MAE': metrics['mae'],
           'R²': metrics['r2'], 'Params': model.count_params(),
           'Time (s)': elapsed, 'Epochs': len(history.history['loss'])}
    all_grid_results.append(row)

    if metrics['rmse'] < best_rmse:
        best_rmse = metrics['rmse']
        best_key = run_name
        all_results['Raw_LSTM'] = {
            'metrics': metrics, 'time': elapsed,
            'epochs': len(history.history['loss']),
            'y_pred': y_pred, 'params': model.count_params(),
            'best_variation': variation,
        }
        all_histories['Raw_LSTM'] = history.history
        model.save(str(MODELS_DIR / "raw_lstm_best.keras"))

    results_manager.log_experiment('DL_Raw', run_name, metrics, {'params': params})

print(f"\n🏆 Melhor LSTM: {best_key} — RMSE={best_rmse:.6f}")

🔵 Grid Search: LSTM com Sinal Raw

--- [1/18] Raw_LSTM_g0: {'dropout_rate': 0.2, 'l2_reg': 0.0001, 'units': [64, 32]}



Epoch 15: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.



Epoch 30: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.



Epoch 71: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.



Epoch 82: ReduceLROnPlateau reducing learning rate to 6.25000029685907e-05.


...


Epoch 43: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.



Epoch 50: ReduceLROnPlateau reducing learning rate to 6.25000029685907e-05.


Epoch 51: early stopping


Restoring model weights from the end of the best epoch: 36.


    RMSE=0.437147  MAE=0.330960  R²=0.810903  Time=704.3s

🏆 Melhor LSTM: Raw_LSTM_g0 — RMSE=0.287276


## 5. Experimento 3: CNN-LSTM

In [6]:
print("="*70)
print("🔵 Grid Search: CNN-LSTM com Sinal Raw")
print("="*70)

grid = generate_dl_grid('CNN_LSTM')
base_params = DL_MODELS_CONFIG['CNN_LSTM'].copy()
best_rmse, best_key = float('inf'), None

for gi, variation in enumerate(grid):
    params = {**base_params, **variation}
    run_name = f"Raw_CNN_LSTM_g{gi}"
    print(f"\n--- [{gi+1}/{len(grid)}] {run_name}: {variation}")

    tf.keras.backend.clear_session()
    with strategy.scope():
        model = create_cnn_lstm_model(input_shape, params=params)

    model_path = str(MODELS_DIR / f"raw_cnn_lstm_g{gi}.keras")
    callbacks = get_callbacks(
        model_path,
        patience_early=training_config['early_stopping_patience'],
        patience_lr=training_config['reduce_lr_patience'],
        min_lr=training_config['min_lr'],
    )

    t0 = time.time()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=training_config['epochs'],
        batch_size=training_config['batch_size'],
        callbacks=callbacks, verbose=0,
    )
    elapsed = time.time() - t0

    y_pred = model.predict(X_test, verbose=0).flatten()
    metrics = evaluator.evaluate(y_test, y_pred)

    print(f"    RMSE={metrics['rmse']:.6f}  MAE={metrics['mae']:.6f}  "
          f"R²={metrics['r2']:.6f}  Time={elapsed:.1f}s")

    row = {'Model': 'Raw_CNN_LSTM', 'grid_idx': gi, **variation,
           'RMSE': metrics['rmse'], 'MAE': metrics['mae'],
           'R²': metrics['r2'], 'Params': model.count_params(),
           'Time (s)': elapsed, 'Epochs': len(history.history['loss'])}
    all_grid_results.append(row)

    if metrics['rmse'] < best_rmse:
        best_rmse = metrics['rmse']
        best_key = run_name
        all_results['Raw_CNN_LSTM'] = {
            'metrics': metrics, 'time': elapsed,
            'epochs': len(history.history['loss']),
            'y_pred': y_pred, 'params': model.count_params(),
            'best_variation': variation,
        }
        all_histories['Raw_CNN_LSTM'] = history.history
        model.save(str(MODELS_DIR / "raw_cnn_lstm_best.keras"))

    results_manager.log_experiment('DL_Raw', run_name, metrics, {'params': params})

print(f"\n🏆 Melhor CNN-LSTM: {best_key} — RMSE={best_rmse:.6f}")

🔵 Grid Search: CNN-LSTM com Sinal Raw

--- [1/36] Raw_CNN_LSTM_g0: {'dropout_rate': 0.2, 'l2_reg': 0.0001, 'cnn_filters': [32, 64], 'lstm_units': [64, 32]}



Epoch 23: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.



Epoch 31: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.



Epoch 38: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.


Epoch 39: early stopping


...


Epoch 75: ReduceLROnPlateau reducing learning rate to 3.125000148429535e-05.



Epoch 82: ReduceLROnPlateau reducing learning rate to 1.5625000742147677e-05.


Epoch 83: early stopping


Restoring model weights from the end of the best epoch: 68.


    RMSE=0.276400  MAE=0.215608  R²=0.924403  Time=697.2s

🏆 Melhor CNN-LSTM: Raw_CNN_LSTM_g26 — RMSE=0.240050


## 6. Experimento 4: Transformer

In [7]:
print("="*70)
print("🔵 Grid Search: Transformer com Sinal Raw")
print("="*70)

grid = generate_dl_grid('Transformer')
base_params = DL_MODELS_CONFIG['Transformer'].copy()
best_rmse, best_key = float('inf'), None

for gi, variation in enumerate(grid):
    params = {**base_params, **variation}
    run_name = f"Raw_Transformer_g{gi}"
    print(f"\n--- [{gi+1}/{len(grid)}] {run_name}: {variation}")

    tf.keras.backend.clear_session()
    with strategy.scope():
        model = create_transformer_model(input_shape, params=params)

    model_path = str(MODELS_DIR / f"raw_transformer_g{gi}.keras")
    callbacks = get_callbacks(
        model_path,
        patience_early=training_config['early_stopping_patience'],
        patience_lr=training_config['reduce_lr_patience'],
        min_lr=training_config['min_lr'],
        use_reduce_lr=not params.get('use_warmup', True),
    )

    t0 = time.time()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=training_config['epochs'],
        batch_size=training_config['batch_size'],
        callbacks=callbacks, verbose=0,
    )
    elapsed = time.time() - t0

    y_pred = model.predict(X_test, verbose=0).flatten()
    metrics = evaluator.evaluate(y_test, y_pred)

    print(f"    RMSE={metrics['rmse']:.6f}  MAE={metrics['mae']:.6f}  "
          f"R²={metrics['r2']:.6f}  Time={elapsed:.1f}s")

    row = {'Model': 'Raw_Transformer', 'grid_idx': gi, **variation,
           'RMSE': metrics['rmse'], 'MAE': metrics['mae'],
           'R²': metrics['r2'], 'Params': model.count_params(),
           'Time (s)': elapsed, 'Epochs': len(history.history['loss'])}
    all_grid_results.append(row)

    if metrics['rmse'] < best_rmse:
        best_rmse = metrics['rmse']
        best_key = run_name
        all_results['Raw_Transformer'] = {
            'metrics': metrics, 'time': elapsed,
            'epochs': len(history.history['loss']),
            'y_pred': y_pred, 'params': model.count_params(),
            'best_variation': variation,
        }
        all_histories['Raw_Transformer'] = history.history
        model.save(str(MODELS_DIR / "raw_transformer_best.keras"))

    results_manager.log_experiment('DL_Raw', run_name, metrics, {'params': params})

print(f"\n🏆 Melhor Transformer: {best_key} — RMSE={best_rmse:.6f}")

🔵 Grid Search: Transformer com Sinal Raw

--- [1/48] Raw_Transformer_g0: {'dropout_rate': 0.15, 'num_heads': 2, 'ff_dim': 64, 'num_transformer_blocks': 2, 'l2_reg': 0.0001}


Epoch 49: early stopping


Restoring model weights from the end of the best epoch: 34.


    RMSE=0.205240  MAE=0.160667  R²=0.958318  Time=423.6s

--- [2/48] Raw_Transformer_g1: {'dropout_rate': 0.15, 'num_heads': 2, 'ff_dim': 64, 'num_transformer_blocks': 2, 'l2_reg': 0.001}


Epoch 50: early stopping


...

    RMSE=0.220660  MAE=0.170379  R²=0.951819  Time=247.7s

--- [18/48] Raw_Transformer_g17: {'dropout_rate': 0.2, 'num_heads': 2, 'ff_dim': 64, 'num_transformer_blocks': 2, 'l2_reg': 0.001}


Epoch 38: early stopping


Restoring model weights from the end of the best epoch: 23.


    RMSE=0.197249  MAE=0.152820  R²=0.961500  Time=338.1s

--- [19/48] Raw_Transformer_g18: {'dropout_rate': 0.2, 'num_heads': 2, 'ff_dim': 64, 'num_transformer_blocks': 3, 'l2_reg': 0.0001}


Epoch 35: early stopping


## 7. Comparação dos Resultados

In [ ]:
# Criar DataFrame comparativo (melhor de cada arquitetura)
comparison_data = []
for model_name, result in all_results.items():
    row = {
        'Model': model_name,
        'RMSE': result['metrics']['rmse'],
        'MAE': result['metrics']['mae'],
        'R²': result['metrics']['r2'],
        'Params': result['params'],
        'Time (s)': result['time'],
        'Epochs': result['epochs']
    }
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('RMSE')

print("\n" + "="*70)
print("📊 COMPARAÇÃO FINAL - Deep Learning com Sinal Raw (melhor por arquitetura)")
print("="*70)
print(comparison_df.to_string(index=False))

# Salvar comparação (melhor por modelo)
out_dir = RESULTS_DIR / "dl_raw_experiments"
comparison_df.to_csv(out_dir / "comparison_dl_raw.csv", index=False)

# Salvar TODOS os resultados do grid
grid_df = pd.DataFrame(all_grid_results)
grid_df.to_csv(out_dir / "all_grid_results_dl_raw.csv", index=False)
print(f"\n✅ CSV comparação salvo: {out_dir / 'comparison_dl_raw.csv'}")
print(f"✅ CSV grid completo salvo: {out_dir / 'all_grid_results_dl_raw.csv'}")
print(f"   Total de combinações treinadas: {len(grid_df)}")

In [ ]:
# Visualização comparativa
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics_to_plot = ['RMSE', 'MAE', 'R²']
colors = plt.cm.tab10.colors

for idx, metric in enumerate(metrics_to_plot):
    data = comparison_df.set_index('Model')[metric].sort_values(
        ascending=(metric != 'R²')
    )
    bars = axes[idx].barh(data.index, data.values, color=colors[:len(data)])
    axes[idx].set_xlabel(metric)
    axes[idx].set_title(f'Comparação: {metric}')
    axes[idx].grid(True, alpha=0.3, axis='x')
    
    for bar, val in zip(bars, data.values):
        axes[idx].text(val, bar.get_y() + bar.get_height()/2,
                      f'{val:.4f}', va='center', ha='left', fontsize=9)

plt.suptitle('Deep Learning com Sinal Raw', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / "dl_raw_experiments" / "comparison_dl_raw.png", dpi=150, bbox_inches='tight')
plt.show()

## 8. Evolução do Treinamento

Visualização detalhada da evolução do processo de treinamento para cada arquitetura:
- **Loss** (Train vs Validation) ao longo das épocas
- **Convergência** — velocidade e estabilidade
- **Early Stopping** — ponto de parada otimizado

In [ ]:
# ── Evolução do Treinamento: Loss (Train vs Val) por modelo ──
n_models = len(all_histories)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

colors = {'loss': '#2196F3', 'val_loss': '#F44336'}

for idx, (model_name, history) in enumerate(all_histories.items()):
    ax = axes[idx]
    epochs_range = range(1, len(history['loss']) + 1)
    
    # Train & Val loss
    ax.plot(epochs_range, history['loss'], color=colors['loss'],
            linewidth=2, label='Train Loss', alpha=0.9)
    ax.plot(epochs_range, history['val_loss'], color=colors['val_loss'],
            linewidth=2, label='Val Loss', alpha=0.9)
    
    # Marcar melhor época (menor val_loss)
    best_epoch = np.argmin(history['val_loss']) + 1
    best_val = min(history['val_loss'])
    ax.axvline(x=best_epoch, color='green', linestyle='--', alpha=0.7, label=f'Best Epoch={best_epoch}')
    ax.scatter([best_epoch], [best_val], color='green', s=100, zorder=5, edgecolors='black')
    
    # Anotação
    ax.annotate(f'Val Loss={best_val:.6f}',
                xy=(best_epoch, best_val),
                xytext=(best_epoch + len(epochs_range)*0.05, best_val * 1.15),
                fontsize=9, color='green',
                arrowprops=dict(arrowstyle='->', color='green', alpha=0.7))
    
    ax.set_xlabel('Época', fontsize=11)
    ax.set_ylabel('Loss (MSE)', fontsize=11)
    ax.set_title(f'{model_name}', fontsize=13, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_yscale('log')

# Esconder subplots não usados
for idx in range(n_models, 4):
    axes[idx].set_visible(False)

plt.suptitle('Evolução do Treinamento — DL Raw (Loss por Época)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / "dl_raw_experiments" / "training_evolution.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Comparativo: todas as curvas de val_loss sobrepostas ──
fig, ax = plt.subplots(figsize=(12, 6))
cmap = plt.cm.tab10
for idx, (model_name, history) in enumerate(all_histories.items()):
    epochs_range = range(1, len(history['val_loss']) + 1)
    ax.plot(epochs_range, history['val_loss'], linewidth=2, label=model_name,
            color=cmap(idx), alpha=0.85)
    best_ep = np.argmin(history['val_loss']) + 1
    best_vl = min(history['val_loss'])
    ax.scatter([best_ep], [best_vl], color=cmap(idx), s=80, zorder=5, edgecolors='black')

ax.set_xlabel('Época', fontsize=12)
ax.set_ylabel('Validation Loss (MSE)', fontsize=12)
ax.set_title('Comparativo — Convergência Val Loss (DL Raw)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_yscale('log')
plt.tight_layout()
plt.savefig(RESULTS_DIR / "dl_raw_experiments" / "val_loss_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Resumo numérico da evolução ──
print("\n📊 Resumo da Evolução do Treinamento:")
print(f"{'Modelo':<20} {'Épocas':>7} {'Train Loss Final':>16} {'Val Loss Final':>15} {'Best Val Loss':>14} {'Best Epoch':>11}")
print("-" * 90)
for model_name, history in all_histories.items():
    n_ep = len(history['loss'])
    best_ep = np.argmin(history['val_loss']) + 1
    print(f"{model_name:<20} {n_ep:>7} {history['loss'][-1]:>16.6f} {history['val_loss'][-1]:>15.6f} {min(history['val_loss']):>14.6f} {best_ep:>11}")

## 9. Análise de Predições

In [ ]:
# Encontrar melhor modelo
best_model_name = comparison_df.iloc[0]['Model']
best_result = all_results[best_model_name]

print(f"\n🏆 Melhor Modelo: {best_model_name}")

# Plot de predições
fig = visualizer.plot_prediction_comparison(
    y_test, best_result['y_pred'],
    model_name=best_model_name,
    n_samples=500,
    save_path=RESULTS_DIR / "dl_raw_experiments" / f"predictions_{best_model_name}.png"
)
plt.show()

## 10. Resumo

In [ ]:
print("\n" + "="*70)
print("📋 RESUMO - Experimentos DL com Sinal Raw")
print("="*70)
print(f"\n✅ Modelos avaliados: {len(all_results)}")
print(f"✅ Melhor modelo: {best_model_name}")
print(f"✅ Melhor RMSE: {comparison_df.iloc[0]['RMSE']:.6f}")
print(f"✅ Melhor R²: {comparison_df.iloc[0]['R²']:.6f}")
print(f"\n📁 Resultados salvos em: {RESULTS_DIR / 'dl_raw_experiments'}")
print("\n🎉 Notebook concluído com sucesso!")